In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [ ]:
import nltk
nltk.download('punkt'),  nltk.download('punkt_tab')

# Train

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from itertools import chain
import click
import os

from ruamel.yaml import YAML

# Загрузка данных

In [ ]:
from src.train import load_external_data, enc_classes

# получаем данные митра
conf = YAML().load(open('params.yaml'))

df = load_external_data(conf)
df, mlb, mlb_ttp = enc_classes(df, conf, use_rare_ttp=False)

# на самом деле 208 train тут уже есть - синтетика
df['split'] = df['split'].fillna('tr')

# train

## nttp train

In [ ]:
from src.funcs import set_seed
from src.spec_nn_funcs import train_bert
from src.aug_sent import add_aug_sents

conf = YAML().load(open('params.yaml'))
set_seed(conf['seed'])
# до TextModelClass, где нейронка инициализируется
from src.spec_nn_funcs import TextDFDataset, TextModelClass, train_eval_bert

# conf = YAML().load(open('params.yaml'))

conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))


model_bert, loss_bert_d, thresh_l, _, _, _, (p_tr_micro, r_tr_micro, f1_tr_micro, p_tr_macro, r_tr_macro, f1_tr_macro) = train_bert(df, mlb, conf, conf_bert, target_col= 'labels', thresh_space_l=[])

## nttp pred

In [ ]:
from transformers import BertTokenizer
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

from torch.utils.data import DataLoader, Dataset

from src.funcs import get_preds


def predict(pred_df, model, thresh_l, conf_bert, suf):
    
    batch_size = conf_bert['nn']['batch_size']
    
    bert_type = conf_bert['nn_bert']['bert_type']
    
    MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']
    
    # model_bert
    
    if bert_type == 'secbert_plus':
        
        checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
        tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
        tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}
        
    elif bert_type == 'secbert':
        checkpoint = 'data/external/models/SecBERT/snapshots/7c603df5bc4c5ba9c731bcc2ea0ab2db36e104cb'
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}
    
    elif bert_type == 'scibert':
        # checkpoint = 'allenai/scibert_scivocab_uncased'
        checkpoint = 'data/external/models/scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1'
        tokenizer = BertTokenizer.from_pretrained(checkpoint, max_length=512)
        tokenizer_opts = {'return_tensors':"pt", 'truncation':True,
                          'padding':'max_length', 'max_length':MAX_SEQ_LENGTH}
    
    
    
    ds = TextDFDataset(pred_df, tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
    ld = DataLoader(ds, batch_size = batch_size, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
    

    Y_pred_proba = np.array(get_preds(model, ld=ld)['pred'])

    pred_df[f'proba_{suf}'] = Y_pred_proba.tolist()
    pred_df[f'pred_{suf}'] = pred_df[f'proba_{suf}'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_l)])
    return pred_df


pred_df = predict(df[['sentence']].assign(target=1), model_bert, thresh_l, conf_bert, suf='labels')
pred_df.head()

In [ ]:

pred_true_df = pred_df.copy().assign(target=df.target)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

In [ ]:
pred_df.head(2)

## ttp train

In [ ]:
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf = YAML().load(open('params.yaml'))
# NEED?
set_seed(conf['seed'])

# чтобы новый конф работал вместо старого в функции
conf_ttp['feat_gen'] = conf_ttp['feat_gen_ttp'] 
conf_ttp['seed'] = conf['seed']
conf_ttp['use_only_proc'] = conf['use_only_proc']

mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])

df_ttp = add_aug_sents(df, conf_ttp, conf_bert_ttp['nn_ttp']['maxlen'])

df_ttp['target'] = mlb_ttp.transform(df_ttp['ttp']).tolist()

In [ ]:
conf = YAML().load(open('params.yaml'))

conf['feat_gen'] = conf_ttp['feat_gen_ttp']
conf['train_eval_model'] = conf_ttp['train_eval_model_ttp']


conf_bert_ttp['nn'] = conf_bert_ttp['nn_ttp']
conf_bert_ttp['nn_bert'] = conf_bert_ttp['nn_bert_ttp']


model_bert_ttp, loss_bert_ttp_d, thresh_ttp_l, _, _, _, (p_tr_micro, r_tr_micro, f1_tr_micro, p_tr_macro, r_tr_macro, f1_tr_macro) = train_bert(df_ttp, mlb_ttp, conf, conf_bert_ttp, target_col= 'ttp', 
                                                                                                                                                thresh_space_l=np.arange(0.001, 1, 0.002))

In [ ]:
pred_df = predict(pred_df, model_bert_ttp, thresh_ttp_l, conf_bert_ttp, suf='ttp')
pred_df.head()

In [ ]:

pred_true_df = pred_df.copy().assign(target=df.target)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])
data['labels'] = data['labels'].map(lambda x: eval(x))
data['origin_labels'] = data['origin_labels'].map(lambda x: eval(x))
data['origin_ttp'] = data['origin_ttp'].map(lambda x: eval(x))


In [ ]:
df.shape, data.shape, df.index, data.index

## тест идентичности основных полей после загрузки

In [ ]:

data[['sentence', 'labels', 'url', 'par_name', 'is_proc']].explode('labels').compare(df[['sentence', 'labels', 'url', 'par_name', 'is_proc']].explode('labels'))

In [ ]:
data[['sentence', 'origin_ttp', 'url', 'par_name', 'is_proc']].explode('origin_ttp').reset_index(drop=True).\
compare(df[['sentence', 'origin_ttp', 'url', 'par_name', 'is_proc']].explode('origin_ttp').reset_index(drop=True))


In [ ]:
data[['sentence', 'origin_labels', 'url', 'par_name', 'is_proc']].explode('origin_labels').reset_index(drop=True).\
compare(df[['sentence', 'origin_labels', 'url', 'par_name', 'is_proc']].explode('origin_labels').reset_index(drop=True))


# код по добавлению энкодинга

In [ ]:
df_s = df.copy()

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

use_rare_ttp = False
mlb_ttp = MultiLabelBinarizer()


if use_rare_ttp:
    ttp_counts_thresh = conf['prep_text']['ttp_counts_thresh']
    ttp_l = df['origin_labels'].explode('origin_labels').value_counts().loc[lambda x: x>ttp_counts_thresh].index.tolist()    
    mlb_ttp.fit([[c] for c in ttp_l+['rare']])
    df['ttp'] = df['origin_labels'].map(lambda x: [it if it in ttp_l else 'rare' for it in x] )
else:
    ttp_l = df['origin_labels'].explode('origin_labels').value_counts().index.tolist()    
    mlb_ttp.fit([[c] for c in ttp_l])
    df['ttp'] = df['origin_labels']

In [ ]:
CLASSES = df.explode('labels')['labels'].dropna().unique()

mlb = MultiLabelBinarizer(classes=CLASSES)
mlb.fit([[c] for c in CLASSES])

df['target'] = mlb.transform(df['labels']).tolist()


df['target_ttp'] = mlb_ttp.transform(df['ttp']).tolist()

# joblib.dump(mlb, conf['prep_text']['mlb_fn'])
# joblib.dump(mlb_ttp, conf['prep_text']['ttp_mlb_fn'])

In [ ]:

sub_l = df['origin_ttp'].explode('origin_ttp').value_counts().loc[lambda x: x>0].index.tolist()

mlb_split = MultiLabelBinarizer()
mlb_split.fit([[c] for c in sub_l+['rare']])

df['origin_ttp_enc'] = mlb_split.transform(df['origin_ttp'].map(lambda x: [it if it in sub_l else 'rare' for it in x] )).tolist()

- train:
    - набор текстов:
        - предварительная обработка специальная:
            - добавление chatgpt для обучения
            - митре + tram + cyberthreats + chatgrpt + sentence_bert_gen
      
        - предварительная обработка общая:
            - токенизация
            - формирование датасетов и лоадеров
        
        - обучение 2 моделей 
        - генетика поверх нескольких прогнозов

    
- predict:
    - набор текстов:
        - предварительная обработка общая

## manual

# Predict